In [ ]:
import sys
sys.path.insert(0, "../config")
sys.path.insert(0, "..")
from config_loader import cfg

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
import logging
logging.getLogger("tensorflow").setLevel(logging.CRITICAL)
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.filterwarnings("ignore", category=ValueWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="Do not pass an `input_shape`")
warnings.filterwarnings("ignore", message="triggered tf.function retracing")
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")
warnings.filterwarnings("ignore", message="Maximum Likelihood optimization failed")
warnings.filterwarnings("ignore", message="A date index has been provided")
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="statsmodels")
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.filterwarnings("ignore", category=ValueWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="Model is not converging")

In [ ]:
from src.backtest.optimize import optimize_all
import pandas as pd

df = pd.read_parquet(cfg.data_path("feature_engineered"))
storage = f"sqlite:///{cfg.model_path('optuna_db')}"

# Zum Löschen bestehender Studies:
#import optuna
#optuna.delete_study(study_name="opt_MSM", storage=storage)
#optuna.delete_study(study_name="opt_HMM", storage=storage)
#optuna.delete_study(study_name="opt_LSTM", storage=storage)
#optuna.delete_study(study_name="opt_Transformer", storage=storage)

# Alle Modelle in einem Rutsch — n_trials und every_nth_fold werden pro Modell
# aus config.yaml gelesen (optimization.n_trials_per_model bzw.
# every_nth_fold_per_model). Thesis-Default: 50 Trials / every_nth_fold=2 für
# MSM & HMM, 30 Trials / every_nth_fold=5 für LSTM & Transformer.
studies = optimize_all(
    df, cfg,
    models=["MSM", "HMM", "LSTM", "Transformer"],
    storage=storage,
)

In [ ]:
from src.backtest.plots import save_optuna_plots

for name, study in studies.items():
    save_optuna_plots(study, name, cfg)

In [ ]:
# Best-Params im Notebook anzeigen UND als .md unter assets/ persistieren
import yaml
from src.backtest.optimize import save_optuna_best_params

for name, study in studies.items():
    print(f"\n# {name}:")
    print(yaml.dump(study.best_params, default_flow_style=False))

save_optuna_best_params(studies, cfg)